### 8 · Hybrid Search — BM25 + Vector Ensemble

Vector search finds **semantically similar** results, but misses exact keyword matches.
BM25 (keyword search) finds **exact terms** but misses semantic meaning.
Hybrid search combines both — best of both worlds.

### What you'll learn
```
1. BM25 keyword retrieval — how it works, when it wins
2. Ensemble retriever — weighted combination of BM25 + vector
3. Reciprocal Rank Fusion (RRF) — how results are merged
4. When hybrid beats pure vector search
```

### When does vector search fail?
- Exact names: "What does `editorSlice` do?" — embeddings may not match the exact token
- Abbreviations: "What is CMS2?" — vector search may match wrong concepts
- Code patterns: "`useReducer` vs `useState`" — keyword match is more reliable

In [ ]:
import json
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv(override=True)

VECTORSTORE_DIR = Path("../data/vectorstore")
CHUNKS_PATH = Path("../data/processed/chunks_enriched.json")
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vectorstore = FAISS.load_local(
    str(VECTORSTORE_DIR), embeddings, allow_dangerous_deserialization=True
)
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

# Load documents for BM25 (needs the raw text)
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    raw_chunks = json.load(f)

documents = [
    Document(page_content=c["page_content"], metadata=c["metadata"])
    for c in raw_chunks
]

print(f"Loaded {vectorstore.index.ntotal} vectors + {len(documents)} documents for BM25.")

---
### Step 1 — BM25 Retriever

BM25 (Best Matching 25) is a classic keyword-based ranking function.
It scores documents by **term frequency** and **inverse document frequency**.

- Doesn't need embeddings or API calls
- Runs entirely in-memory
- Great for exact term matching

In [ ]:
# Create BM25 retriever from the same documents
bm25_retriever = BM25Retriever.from_documents(documents, k=5)

# Also create a vector retriever for comparison
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("BM25 and vector retrievers ready.")

In [ ]:
# ── Compare: Where BM25 wins ──────────────────────────────────────────────────

def compare_retrievers(query: str):
    """Side-by-side comparison of BM25 vs Vector retrieval."""
    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = vector_retriever.invoke(query)

    bm25_sources = [d.metadata.get("source", "?") for d in bm25_docs]
    vector_sources = [d.metadata.get("source", "?") for d in vector_docs]

    print(f"Query: {query}\n")
    print(f"{'BM25 Sources':<50} {'Vector Sources'}")
    print("-" * 95)
    for i in range(max(len(bm25_sources), len(vector_sources))):
        bm = bm25_sources[i] if i < len(bm25_sources) else ""
        vec = vector_sources[i] if i < len(vector_sources) else ""
        print(f"{bm:<50} {vec}")

    # Overlap
    bm25_set = set(d.page_content[:100] for d in bm25_docs)
    vector_set = set(d.page_content[:100] for d in vector_docs)
    overlap = len(bm25_set & vector_set)
    print(f"\nOverlap: {overlap}/{len(bm25_docs)} chunks in common")
    print()


# Exact term — BM25 should win
compare_retrievers("What does editorSlice do?")

# Semantic — Vector should win
compare_retrievers("How is application state kept in sync?")

# Mixed — both should contribute
compare_retrievers("What Firestore collections does actiontype-wip use?")

---
### Step 2 — Ensemble Retriever (Reciprocal Rank Fusion)

The **Ensemble Retriever** runs multiple retrievers in parallel and merges results
using **Reciprocal Rank Fusion (RRF)**:

```
RRF_score(doc) = sum( 1 / (k + rank_in_retriever_i) ) for each retriever
```

Documents that rank high in **both** retrievers get boosted.
The `weights` parameter controls how much each retriever contributes.

In [ ]:
# ── Manual Ensemble (since EnsembleRetriever may not be available) ────────────
# We implement RRF ourselves — this is what the framework does internally.

def reciprocal_rank_fusion(
    results_list: list[list[Document]],
    weights: list[float] | None = None,
    k: int = 60,
    top_n: int = 5,
) -> list[Document]:
    """
    Merge results from multiple retrievers using Reciprocal Rank Fusion.

    Args:
        results_list: List of result lists from each retriever
        weights: Weight per retriever (default: equal)
        k: RRF constant (default 60)
        top_n: Number of results to return
    """
    if weights is None:
        weights = [1.0] * len(results_list)

    # Score each document
    doc_scores: dict[str, float] = {}
    doc_map: dict[str, Document] = {}

    for results, weight in zip(results_list, weights):
        for rank, doc in enumerate(results):
            # Use content hash as key for deduplication
            doc_key = doc.page_content[:200]
            rrf_score = weight * (1.0 / (k + rank + 1))

            if doc_key in doc_scores:
                doc_scores[doc_key] += rrf_score
            else:
                doc_scores[doc_key] = rrf_score
                doc_map[doc_key] = doc

    # Sort by fused score
    sorted_keys = sorted(doc_scores.keys(), key=lambda x: doc_scores[x], reverse=True)
    return [doc_map[key] for key in sorted_keys[:top_n]]


print("RRF implementation ready.")

In [ ]:
# ── Hybrid Retriever Function ─────────────────────────────────────────────────

def hybrid_retrieve(
    query: str,
    vector_weight: float = 0.5,
    bm25_weight: float = 0.5,
    top_n: int = 5,
) -> list[Document]:
    """Run BM25 + vector search and merge with RRF."""
    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = vector_retriever.invoke(query)

    fused = reciprocal_rank_fusion(
        [bm25_docs, vector_docs],
        weights=[bm25_weight, vector_weight],
        top_n=top_n,
    )
    return fused


# Test it
query = "What does editorSlice do?"
hybrid_docs = hybrid_retrieve(query)

print(f"Query: {query}\n")
print(f"Hybrid results ({len(hybrid_docs)} docs):")
for i, doc in enumerate(hybrid_docs):
    print(f"  [{i}] {doc.metadata.get('source','?'):<45} {doc.metadata.get('section','?')[:40]}")

---
### Step 3 — Full Comparison: Vector vs BM25 vs Hybrid

Run all three retrievers on a set of test queries and compare.

In [ ]:
TEST_QUERIES = [
    "What does editorSlice do?",                                # exact term
    "How is application state kept in sync?",                   # semantic
    "What Firestore collections does actiontype-wip use?",      # mixed
    "What is the dagre layout algorithm used for?",             # specific library
    "How do I avoid anti-patterns in Redux?",                   # semantic + keyword
]

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using ONLY the provided context. Be concise.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])
answer_chain = ANSWER_PROMPT | llm | StrOutputParser()

for query in TEST_QUERIES:
    print(f"\nQuery: {query}")
    print("=" * 90)

    v_docs = vector_retriever.invoke(query)
    b_docs = bm25_retriever.invoke(query)
    h_docs = hybrid_retrieve(query)

    v_sources = set(d.metadata.get("source", "?") for d in v_docs)
    b_sources = set(d.metadata.get("source", "?") for d in b_docs)
    h_sources = set(d.metadata.get("source", "?") for d in h_docs)

    print(f"  Vector sources:  {sorted(v_sources)}")
    print(f"  BM25 sources:    {sorted(b_sources)}")
    print(f"  Hybrid sources:  {sorted(h_sources)}")
    print(f"  Unique to BM25:  {sorted(b_sources - v_sources)}")
    print(f"  Unique to Vector:{sorted(v_sources - b_sources)}")

---
### Step 4 — Tuning Weights

The balance between BM25 and vector search depends on your data:
- **Technical docs with code** → higher BM25 weight (0.6)
- **Natural language docs** → higher vector weight (0.6)
- **Mixed** → equal weights (0.5/0.5)

In [ ]:
# ── Weight experiment ─────────────────────────────────────────────────────────

query = "What does editorSlice do?"
weight_configs = [
    (0.0, 1.0, "Pure BM25"),
    (0.3, 0.7, "BM25-heavy (0.7 BM25)"),
    (0.5, 0.5, "Equal (0.5 / 0.5)"),
    (0.7, 0.3, "Vector-heavy (0.7 vector)"),
    (1.0, 0.0, "Pure Vector"),
]

print(f"Query: {query}\n")
print(f"{'Config':<30} Sources")
print("-" * 90)

for v_w, b_w, label in weight_configs:
    docs = hybrid_retrieve(query, vector_weight=v_w, bm25_weight=b_w)
    sources = [d.metadata.get("source", "?") for d in docs]
    print(f"{label:<30} {sources}")

---
### Step 5 — Hybrid Search in LangGraph

Integrate hybrid retrieval as a single node in the graph.

In [ ]:
from typing import TypedDict
from langgraph.graph import END, StateGraph

class GraphState(TypedDict):
    question: str
    documents: list[Document]
    answer: str

def hybrid_retrieve_node(state: GraphState) -> dict:
    docs = hybrid_retrieve(state["question"], vector_weight=0.5, bm25_weight=0.5)
    print(f"  [hybrid_retrieve] → {len(docs)} docs")
    return {"documents": docs}

def generate_answer(state: GraphState) -> dict:
    context = "\n\n---\n\n".join(
        f"[{d.metadata.get('source', '?')}]\n{d.page_content}"
        for d in state["documents"]
    )
    answer = answer_chain.invoke({"question": state["question"], "context": context})
    print(f"  [generate] → {len(answer)} chars")
    return {"answer": answer}

workflow = StateGraph(GraphState)
workflow.add_node("hybrid_retrieve", hybrid_retrieve_node)
workflow.add_node("generate_answer", generate_answer)
workflow.set_entry_point("hybrid_retrieve")
workflow.add_edge("hybrid_retrieve", "generate_answer")
workflow.add_edge("generate_answer", END)

hybrid_graph = workflow.compile()

# Test
result = hybrid_graph.invoke({"question": "What does editorSlice do?"})
print(f"\nAnswer: {result['answer']}")

---
### Takeaways

| Concept | What you learned |
|---------|------------------|
| **BM25** | Keyword-based retrieval, great for exact terms, no API cost |
| **Reciprocal Rank Fusion** | Merges ranked lists by combining inverse ranks |
| **Hybrid search** | Combines BM25 + vector for higher recall on mixed queries |
| **Weight tuning** | Technical docs benefit from higher BM25 weight |

**Production upgrades:**
- Use Elasticsearch or Meilisearch for production BM25 (in-memory BM25 doesn't scale)
- Combine hybrid search with reranking (notebook 7) for best results
- Auto-tune weights using your eval dataset (notebook 5)